# Start Partition Comparison

This notebook compares the available start partition algorithms for the Dense Graph Partition experiments.

The comparison is performed separately for:

- Powerlaw and Erdős–Rényi graphs,
- sparse and dense instances,
- small and large instances (if both are provided).

Only two aggregated metrics are reported:

- **mean relative to best**: mean quotient between the best density found on an instance and the density obtained by the algorithm,
- **mean runtime**: mean runtime in seconds.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Configuration

In [2]:
RESULTS_FILE = Path("../../results/experiment1/raw_results.csv")

GRAPH_ORDER = ["powerlaw", "er"]
SIZE_ORDER = ["small", "large"]
REGIME_ORDER = ["sparse", "dense"]
ALGORITHM_ORDER = [
    "singleton",
    "matching",
    "maximum_matching",
    "maximum_matching_edge_cover",
    "high_degree_first_matching",
    "high_degree_product_matching",
    "kapoce",
    "leiden_mdgp",
]

GROUP_ORDER = ["graph_type", "size_class", "regime"]

## Load data

In [3]:
raw = pd.read_csv(RESULTS_FILE)

raw["graph_type"] = pd.Categorical(
    raw["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)

raw["size_class"] = pd.Categorical(
    raw["size_class"],
    categories=SIZE_ORDER,
    ordered=True,
)

raw["regime"] = pd.Categorical(
    raw["regime"],
    categories=REGIME_ORDER,
    ordered=True,
)

raw["algorithm"] = pd.Categorical(
    raw["algorithm"],
    categories=ALGORITHM_ORDER,
    ordered=True,
)

## Solution quality

For every instance, the best result across all start partition algorithms is used as the reference. Relative solution quality is defined as

$
\frac{\text{best solution quality on the instance}}
     {\text{solution quality of the algorithm}}.
$

A value of $1.0$ means that the algorithm matches the best result found on the same instance. Values greater than $1.0$ indicate the remaining quality gap.

In [4]:
instance_keys = [
    "graph_type",
    "size_class",
    "regime",
    "dataset",
    "instance",
]

density_table = raw.pivot_table(
    index=instance_keys,
    columns="algorithm",
    values="density",
    observed=True,
)

density_table = density_table[ALGORITHM_ORDER]
density_table.columns.name = None

In [5]:
best_per_instance = density_table.max(axis=1)

relative_to_best = density_table.rdiv(
    best_per_instance,
    axis=0,
)

relative_to_best.columns.name = "algorithm"

relative_to_best_summary = (
    relative_to_best
    .groupby(level=GROUP_ORDER)
    .mean()
    .stack()
    .rename("mean_relative_to_best")
    .reset_index()
    .sort_values(GROUP_ORDER + ["algorithm"])
    .reset_index(drop=True)
)

relative_to_best_summary

,graph_type,size_class,regime,algorithm,mean_relative_to_best
0,powerlaw,small,sparse,singleton,inf
1,powerlaw,small,sparse,matching,1.410530
2,powerlaw,small,sparse,maximum_matching,1.114775
3,powerlaw,small,sparse,maximum_matching_edge_cover,1.111345
4,powerlaw,small,sparse,high_degree_first_matching,1.489544
...,...,...,...,...,...
59,er,large,dense,maximum_matching_edge_cover,1.330447
60,er,large,dense,high_degree_first_matching,1.375947
61,er,large,dense,high_degree_product_matching,1.375947
62,er,large,dense,kapoce,1.000000


In [6]:
runtime_summary = (
    raw
    .groupby(GROUP_ORDER + ["algorithm"], observed=True, as_index=False,)
    .agg(
        mean_runtime_seconds=("runtime", "mean"),
    )
    .sort_values(GROUP_ORDER + ["algorithm"])
    .reset_index(drop=True)
)

runtime_summary

,graph_type,size_class,regime,algorithm,mean_runtime_seconds
0,powerlaw,small,sparse,singleton,0.000150
1,powerlaw,small,sparse,matching,0.001222
2,powerlaw,small,sparse,maximum_matching,0.054981
3,powerlaw,small,sparse,maximum_matching_edge_cover,0.048525
4,powerlaw,small,sparse,high_degree_first_matching,0.004819
...,...,...,...,...,...
59,er,large,dense,maximum_matching_edge_cover,4.710416
60,er,large,dense,high_degree_first_matching,0.232747
61,er,large,dense,high_degree_product_matching,0.125471
62,er,large,dense,kapoce,0.815004


In [7]:
final_table = relative_to_best_summary.merge(
    runtime_summary,
    on=GROUP_ORDER + ["algorithm"],
)

final_table

,graph_type,size_class,regime,algorithm,mean_relative_to_best,mean_runtime_seconds
0,powerlaw,small,sparse,singleton,inf,0.000150
1,powerlaw,small,sparse,matching,1.410530,0.001222
2,powerlaw,small,sparse,maximum_matching,1.114775,0.054981
3,powerlaw,small,sparse,maximum_matching_edge_cover,1.111345,0.048525
4,powerlaw,small,sparse,high_degree_first_matching,1.489544,0.004819
...,...,...,...,...,...,...
59,er,large,dense,maximum_matching_edge_cover,1.330447,4.710416
60,er,large,dense,high_degree_first_matching,1.375947,0.232747
61,er,large,dense,high_degree_product_matching,1.375947,0.125471
62,er,large,dense,kapoce,1.000000,0.815004


## Empirical approximation bound for KaPoCE

The maximum-cardinality matching construction is a 2-approximation for MDGP. Its objective value therefore provides an upper bound on the unknown optimum.

For every instance, the quality of the KaPoCE solution can consequently be bounded relative to the optimum using the ratio between KaPoCE and maximum matching.

In [8]:
maximum_matching = (
    raw[raw["algorithm"] == "maximum_matching"][instance_keys + ["density"]]
    .rename(columns={"density": "maximum_matching_density"})
)

kapoce = (
    raw[raw["algorithm"] == "kapoce"][instance_keys + ["density"]]
    .rename(columns={"density": "kapoce_density"})
)

approximation_per_instance = maximum_matching.merge(
    kapoce,
    on=instance_keys,
)

approximation_per_instance["kapoce_approximation_bound"] = (
        2 * approximation_per_instance["maximum_matching_density"] / approximation_per_instance["kapoce_density"]
)

In [9]:
approximation_summary = (
    approximation_per_instance
    .groupby(GROUP_ORDER, as_index=False, observed=True)
    .agg(
        max_approximation_bound=("kapoce_approximation_bound", "max")
    )
    .sort_values(GROUP_ORDER)
    .reset_index(drop=True)
)

approximation_summary

,graph_type,size_class,regime,max_approximation_bound
0,powerlaw,small,sparse,1.921647
1,powerlaw,small,dense,1.692109
2,powerlaw,large,sparse,1.898275
3,powerlaw,large,dense,1.646444
4,er,small,sparse,1.811024
5,er,small,dense,1.594297
6,er,large,sparse,1.949223
7,er,large,dense,1.576957


## LaTeX helper functions

In [10]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor


def latex_operator(operator: str) -> str:
    return r"\texttt{" + operator.replace("_", r"\_") + "}"


def format_number(value: float, decimals: int) -> str:
    return f"{value:.{decimals}f}"

## Build LaTeX tables

In [22]:
def make_start_partition_latex_table(df: pd.DataFrame, graph_type: str, caption: str, label: str) -> str:
    graph_df = df[df["graph_type"] == graph_type]

    groups = list(graph_df.groupby(["size_class", "regime"], observed=True, sort=False))

    lines = [
        r"\begin{table}[!htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{p{2cm}p{6cm}rr}",
        r"\toprule",
        r"Dataset & Initial partition & \shortstack{Mean relative\\solution quality} & \shortstack{Mean run\\time (s)} \\",
        r"\midrule",
    ]

    for dataset_index, ((size_class, regime), part) in enumerate(groups):
        best_quality = part["mean_relative_to_best"].min()
        dataset_label = f"{size_class} {regime}"

        for row_index, row in enumerate(part.itertuples(index=False)):
            dataset_cell = (
                rf"\multirow{{{len(part)}}}{{*}}{{{dataset_label}}}"
                if row_index == 0
                else ""
            )

            quality = format_number(row.mean_relative_to_best, 4)

            if np.isclose(row.mean_relative_to_best, best_quality):
                quality = rf"\textbf{{{quality}}}"

            lines.append(
                f"{dataset_cell} "
                f"& {latex_operator(str(row.algorithm))} "
                f"& {quality} "
                f"& {format_number(row.mean_runtime_seconds, 5)} \\\\"
            )

        if dataset_index < len(groups) - 1:
            lines.append(r"\cmidrule(l){1-4}")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [18]:
powerlaw_latex = make_start_partition_latex_table(
    final_table,
    graph_type="powerlaw",
    caption=(
        "Mean relative solution quality and mean run time of the initial partition methods on Powerlaw instances."
    ),
    label="tab:start_partition_powerlaw",
)

print(powerlaw_latex)

\begin{table}[t]
\centering
\caption{Mean relative solution quality and mean run time of the initial partition methods on Powerlaw instances.}
\label{tab:start_partition_powerlaw}
\begin{tabular}{p{2cm}p{6cm}rr}
\toprule
Dataset & Initial partition & \shortstack{Mean relative\\solution quality} & \shortstack{Mean run\\time (s)} \\
\midrule
\multirow{8}{*}{small sparse} & \texttt{singleton} & inf & 0.00015 \\
 & \texttt{matching} & 1.4105 & 0.00122 \\
 & \texttt{maximum\_matching} & 1.1148 & 0.05498 \\
 & \texttt{maximum\_matching\_edge\_cover} & 1.1113 & 0.04852 \\
 & \texttt{high\_degree\_first\_matching} & 1.4895 & 0.00482 \\
 & \texttt{high\_degree\_product\_matching} & 1.4895 & 0.00323 \\
 & \texttt{kapoce} & \textbf{1.0008} & 0.05106 \\
 & \texttt{leiden\_mdgp} & 1.0445 & 0.01025 \\
\cmidrule(l){1-4}
\multirow{8}{*}{small dense} & \texttt{singleton} & inf & 0.00023 \\
 & \texttt{matching} & 1.4310 & 0.00264 \\
 & \texttt{maximum\_matching} & 1.2367 & 0.10271 \\
 & \texttt{maximum\

In [23]:
er_latex = make_start_partition_latex_table(
    final_table,
    graph_type="er",caption=(
        "Mean relative solution quality and mean run time of the initial partition methods on Erdős-Rényi instances."
    ),
    label="tab:start_partition_er",
)

print(er_latex)

\begin{table}[p]
\centering
\caption{Mean relative solution quality and mean run time of the initial partition methods on Erdős-Rényi instances.}
\label{tab:start_partition_er}
\begin{tabular}{p{2cm}p{6cm}rr}
\toprule
Dataset & Initial partition & \shortstack{Mean relative\\solution quality} & \shortstack{Mean run\\time (s)} \\
\midrule
\multirow{8}{*}{small sparse} & \texttt{singleton} & inf & 0.00009 \\
 & \texttt{matching} & 1.2710 & 0.00135 \\
 & \texttt{maximum\_matching} & 1.1662 & 0.07020 \\
 & \texttt{maximum\_matching\_edge\_cover} & 1.1627 & 0.05964 \\
 & \texttt{high\_degree\_first\_matching} & 1.3646 & 0.00644 \\
 & \texttt{high\_degree\_product\_matching} & 1.3646 & 0.00295 \\
 & \texttt{kapoce} & \textbf{1.0000} & 0.06192 \\
 & \texttt{leiden\_mdgp} & 1.1168 & 0.00510 \\
\cmidrule(l){1-4}
\multirow{8}{*}{small dense} & \texttt{singleton} & inf & 0.00008 \\
 & \texttt{matching} & 1.3767 & 0.00117 \\
 & \texttt{maximum\_matching} & 1.3217 & 0.09250 \\
 & \texttt{maximum\_ma

In [28]:
def make_approximation_latex_table(summary: pd.DataFrame) -> str:
    graph_labels = {
        "powerlaw": "Powerlaw",
        "er": "Erdős-Rényi",
    }

    graph_groups = list(summary.groupby("graph_type", observed=True, sort=False))

    lines = [
        r"\begin{table}[H]",
        r"\centering",
        r"\caption{Empirical approximation bounds for the solutions found by KaPoCE, derived from the 2-approximation guarantee of Maximum Matching.}",
        r"\label{tab:kapoce_approximation}",
        r"\begin{tabular}{llr}",
        r"\toprule",
        r"Graph type & Dataset & Approximation bound \\",
        r"\midrule",
    ]

    for graph_index, (graph_type, graph_df) in enumerate(graph_groups):
        for row_index, row in enumerate(graph_df.itertuples(index=False)):
            graph_cell = (
                rf"\multirow{{{len(graph_df)}}}{{*}}{{{graph_labels[graph_type]}}}"
                if row_index == 0
                else ""
            )

            dataset_label = f"{row.size_class} {row.regime}"

            lines.append(
                f"{graph_cell} & {dataset_label} & {format_number(row.max_approximation_bound, 3)} \\\\"
            )

        if graph_index < len(graph_groups) - 1:
            lines.append(r"\midrule")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [27]:
approximation_table = make_approximation_latex_table(approximation_summary)
print(approximation_table)

\begin{table}[H]
\centering
\caption{Empirical approximation bounds for the solutions found by KaPoCE, derived from the 2-approximation guarantee of Maximum Matching.}
\label{tab:kapoce_approximation}
\begin{tabular}{llr}
\toprule
Graph typ & Dataset & Approximation bound \\
\midrule
\multirow{4}{*}{Powerlaw} & small sparse & 1.922 \\
 & small dense & 1.692 \\
 & large sparse & 1.898 \\
 & large dense & 1.646 \\
\midrule
\multirow{4}{*}{Erdős-Rényi} & small sparse & 1.811 \\
 & small dense & 1.594 \\
 & large sparse & 1.949 \\
 & large dense & 1.577 \\
\bottomrule
\end{tabular}
\end{table}
